In [1]:
# Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA


data = pd.read_csv('data/application_train_FE_baked.csv')

In [ ]:
# Define feature sets for different models
target = "TARGET"

# Basic demographic and financial features
predictors1 = [
    "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "debt_ratio",
    "CNT_CHILDREN", "CNT_FAM_MEMBERS", "REGION_POPULATION_RELATIVE",
    "REGION_RATING", "DAYS_BIRTH", "LIVE_CITY_NOT_WORK_CITY"
]

# Previous application statistics
predictors2 = [
    "prev_app_count", "prev_approved_rate", "prev_refused_rate",
    "prev_amt_credit_mean", "prev_cnt_payment_mean", "prev_ann_to_credit_mean",
    "prev_interest_mean", "prev_rate_mean_all", "prev_share_mean_all"
]

# Recent application history features
predictors3 = [
    "prev_last3_n", "prev_last3_approved_rate", "prev_last3_amt_credit_mean",
    "prev_last3_cnt_payment_mean", "prev_last3_ann_to_credit_mean",
    "prev_last5_n", "prev_last5_approved_rate", "prev_last5_cnt_payment_mean"
]

# Document and registration features
predictors4 = [
    "DAYS_REGISTRATION", "DAYS_ID_PUBLISH", "DAYS_LAST_PHONE_CHANGE",
    "FLAG_PHONE", "FLAG_EMAIL", "LIVE_CITY_NOT_WORK_CITY",
    "REGION_POPULATION_RELATIVE", "REGION_RATING"
]

# Contract and property features
predictors5 = [
    "NAME_CONTRACT_TYPE_Cash.loans", "NAME_CONTRACT_TYPE_Revolving.loans",
    "FLAG_OWN_REALTY_Y", "FLAG_OWN_CAR_Y",
    "NAME_HOUSING_TYPE_House...apartment", "NAME_HOUSING_TYPE_Rented.apartment",
    "NAME_HOUSING_TYPE_With.parents", "AMT_CREDIT", "AMT_ANNUITY"
]

# Income and education features
predictors6 = [
    "NAME_INCOME_TYPE_Working", "NAME_INCOME_TYPE_Commercial.associate",
    "NAME_INCOME_TYPE_Pensioner", "NAME_INCOME_TYPE_State.servant",
    "NAME_EDUCATION_TYPE_Secondary...secondary.special",
    "NAME_EDUCATION_TYPE_Higher.education", "NAME_EDUCATION_TYPE_Lower.secondary",
    "AMT_INCOME_TOTAL", "REGION_RATING"
]

# Combined important features
predictors7 = [
    "EXT_SOURCE_2", "AMT_CREDIT", "AMT_ANNUITY", "AMT_INCOME_TOTAL",
    "debt_ratio", "prev_approved_rate", "prev_app_count",
    "DAYS_BIRTH", "DAYS_REGISTRATION", "REGION_RATING"
]

# Most important features subset
predictors8 = [
    "EXT_SOURCE_2", "AMT_CREDIT", "AMT_INCOME_TOTAL", "debt_ratio", "prev_approved_rate"
]

In [9]:
# Make random split, stratified split by target variable, and stratified split by region rating, debt ratio, and target
train_data_random, val_data_random = train_test_split(data, test_size=0.2, random_state=42)
train_data_stratified, val_data_stratified = train_test_split(data, test_size=0.2, random_state=42, stratify=data['TARGET'])

# For custom stratification, create bins for continuous variables first
data['REGION_RATING_bins'] = pd.qcut(data['REGION_RATING'], q=4, labels=['VL', 'L', 'H', 'VH'], duplicates='drop')
data['debt_ratio_bins'] = pd.qcut(data['debt_ratio'], q=4, labels=['VL', 'L', 'H', 'VH'], duplicates='drop')

# Create a combined stratification column
data['strat_col'] = data['REGION_RATING_bins'].astype(str) + '_' + data['debt_ratio_bins'].astype(str) + '_' + data['TARGET'].astype(str)

# Now stratify by the combined column
train_data_custom, val_data_custom = train_test_split(data, test_size=0.2, random_state=42, stratify=data['strat_col'])

ValueError: Bin labels must be one fewer than the number of bin edges

In [ ]:
# Metrics
def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def recall(y_true, y_pred):
    true_positives = np.sum((y_true == 1) & (y_pred == 1))
    possible_positives = np.sum(y_true == 1)
    return true_positives / possible_positives if possible_positives > 0 else 0

def precision(y_true, y_pred):
    true_positives = np.sum((y_true == 1) & (y_pred == 1))
    predicted_positives = np.sum(y_pred == 1)
    return true_positives / predicted_positives if predicted_positives > 0 else 0

def specificity(y_true, y_pred):
    true_negatives = np.sum((y_true == 0) & (y_pred == 0))
    possible_negatives = np.sum(y_true == 0)
    return true_negatives / possible_negatives if possible_negatives > 0 else 0

def f1_score(y_true, y_pred):
    prec = precision(y_true, y_pred)
    rec = recall(y_true, y_pred)
    return 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0

def roc_auc(y_true, y_pred_prob):
    y_true = np.array(y_true, dtype=float)
    y_pred_prob = np.array(y_pred_prob, dtype=float)

    if np.all(np.isin(y_pred_prob, [0, 1])):
        tp = np.sum((y_pred_prob == 1) & (y_true == 1))
        tn = np.sum((y_pred_prob == 0) & (y_true == 0))
        fp = np.sum((y_pred_prob == 1) & (y_true == 0))
        fn = np.sum((y_pred_prob == 0) & (y_true == 1))
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        return 0.5 * (1 + tpr - fpr)
    
    thresholds = np.unique(np.concatenate([np.array([0, 1]), y_pred_prob]))
    thresholds = np.sort(thresholds)[::-1]  # Sort in descending order
    tpr = np.zeros(len(thresholds))
    fpr = np.zeros(len(thresholds))
    
    for i, threshold in enumerate(thresholds):
        tp = np.sum((y_pred_prob >= threshold) & (y_true == 1))
        fn = np.sum((y_pred_prob < threshold) & (y_true == 1))
        fp = np.sum((y_pred_prob >= threshold) & (y_true == 0))
        tn = np.sum((y_pred_prob < threshold) & (y_true == 0))
        tpr[i] = tp / (tp + fn) if (tp + fn) > 0 else 0
        fpr[i] = fp / (fp + tn) if (fp + tn) > 0 else 0

    sort_idx = np.argsort(fpr)
    fpr = fpr[sort_idx]
    tpr = tpr[sort_idx]
    auc = np.abs(np.trapz(tpr, fpr))
    return auc

In [ ]:
# Cross-validation
def cross_validate(model, X, y, cv=5):
    fold_size = len(X) // cv
    metrics = {'accuracy': [], 'recall': [], 'precision': [], 'specificity': [], 'f1_score': [], 'roc_auc': []}
    
    for fold in range(cv):
        start = fold * fold_size
        end = (fold + 1) * fold_size if fold != cv - 1 else len(X)
        
        X_val_fold = X[start:end]
        y_val_fold = y[start:end]
        X_train_fold = np.concatenate([X[:start], X[end:]], axis=0)
        y_train_fold = np.concatenate([y[:start], y[end:]], axis=0)
        
        model.fit(X_train_fold, y_train_fold)
        y_pred = model.predict(X_val_fold)
        print(model.coef_ if hasattr(model, 'coef_') else "No coefficients")
        if hasattr(model, "predict_proba"):
            y_pred_prob = model.predict_proba(X_val_fold)[:, 1]
        else:
            y_pred_prob = model.decision_function(X_val_fold)
            y_pred_prob = (y_pred_prob - y_pred_prob.min()) / (y_pred_prob.max() - y_pred_prob.min())
        
        metrics['accuracy'].append(accuracy(y_val_fold, y_pred))
        metrics['recall'].append(recall(y_val_fold, y_pred))
        metrics['precision'].append(precision(y_val_fold, y_pred))
        metrics['specificity'].append(specificity(y_val_fold, y_pred))
        metrics['f1_score'].append(f1_score(y_val_fold, y_pred))
        metrics['roc_auc'].append(roc_auc(y_val_fold, y_pred_prob))
    
    avg_metrics = {key: np.mean(value) for key, value in metrics.items()}
    return avg_metrics

In [ ]:
# Models

# Logistic Regression
log_reg = LogisticRegression(max_iter=1000, class_weight='balanced')

# SVM
svm_model = SVC(probability=True, class_weight='balanced')

# LDA
lda_model = LDA()


In [ ]:
# Run Models


